# Image Denoising Autoencoder

The autoencoder learns to reconstruct clean MNIST digits from noisy inputs. The encoder compresses each image into a representation and the decoder turns that representation back into pixels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:10000].astype("float32") / 255.0
x_test = x_test[:1000].astype("float32") / 255.0
rng = np.random.default_rng(42)
noise_train = np.clip(x_train + 0.35 * rng.normal(size=x_train.shape), 0, 1)
noise_test = np.clip(x_test + 0.35 * rng.normal(size=x_test.shape), 0, 1)

model = models.Sequential([
	layers.Input(shape=(28, 28)),
	layers.Flatten(),
	layers.Dense(128, activation="relu"),
	layers.Dense(32, activation="relu"),
	layers.Dense(128, activation="relu"),
	layers.Dense(28 * 28, activation="sigmoid"),
	layers.Reshape((28, 28))
])
model.compile(optimizer="adam", loss="mse")
history = model.fit(noise_train, x_train, validation_data=(noise_test, x_test), epochs=3, batch_size=128, verbose=1)
denoised = model.predict(noise_test[:8], verbose=0)
print(f"Final validation loss: {history.history['val_loss'][-1]:.4f}")

fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for index in range(8):
	axes[0, index].imshow(noise_test[index], cmap="gray")
	axes[1, index].imshow(denoised[index], cmap="gray")
	axes[2, index].imshow(x_test[index], cmap="gray")
	for row in range(3):
		axes[row, index].axis("off")
axes[0, 0].set_ylabel("Noisy")
axes[1, 0].set_ylabel("Decoded")
axes[2, 0].set_ylabel("Clean")
plt.tight_layout()
plt.show()